In [1]:
import glob, json
import numpy as np
import plotly.graph_objects as go

def _find_key(d, substrings):
    """Find first key in dict containing any of substrings (case-insensitive)."""
    keys = list(d.keys())
    for s in substrings:
        s = s.lower()
        for k in keys:
            if s in k.lower():
                return k
    return None

def load_bisl_json(prefix="fk_inputs/loc/my_eventset-ev0"):
    """
    Load BISL output JSON produced by infrapy run_loc.
    Returns (filename, obj).
    """
    cands = sorted(
        glob.glob(prefix + "*.json")
        + glob.glob(prefix + "*.loc.json")
        + glob.glob(prefix + "*.bisl.json")
    )
    if not cands:
        raise FileNotFoundError(f"No BISL JSON found for prefix '{prefix}'. Try: ls {prefix}*")
    fn = cands[0]
    with open(fn, "r") as f:
        obj = json.load(f)
    return fn, obj

def extract_bisl_grid_and_point(obj):
    """
    Extract posterior grid (lat_grid, lon_grid, pdf_grid) and a point estimate (est_lon, est_lat).
    Robustly searches for likely fields.
    """
    # Try to find grid arrays at top-level or nested dicts
    def try_extract(d):
        lat_k = _find_key(d, ["lat_grid", "latitude_grid", "latgrid", "lat"])
        lon_k = _find_key(d, ["lon_grid", "longitude_grid", "longrid", "lon"])
        pdf_k = _find_key(d, ["pdf", "posterior", "prob", "likelihood", "pgrid"])
        if lat_k and lon_k and pdf_k:
            lat = np.array(d[lat_k])
            lon = np.array(d[lon_k])
            pdf = np.array(d[pdf_k])
            # Make sure shapes are compatible
            return lat, lon, pdf
        return None

    out = try_extract(obj)
    if out is None:
        for k, v in obj.items():
            if isinstance(v, dict):
                out = try_extract(v)
                if out is not None:
                    break
    if out is None:
        raise KeyError(
            "Couldn't find posterior grid arrays in BISL JSON. "
            f"Top keys: {list(obj.keys())[:40]}"
        )
    lat_grid, lon_grid, pdf_grid = out

    # Extract point estimate (MAP) if present
    est_lat = est_lon = None

    # Look for a MAP dict commonly used
    map_k = _find_key(obj, ["map"])
    if map_k and isinstance(obj[map_k], dict):
        m = obj[map_k]
        lat_k = _find_key(m, ["lat"])
        lon_k = _find_key(m, ["lon"])
        if lat_k and lon_k:
            est_lat = float(m[lat_k])
            est_lon = float(m[lon_k])

    # If MAP not found, try generic estimate keys
    if est_lat is None or est_lon is None:
        lat_k = _find_key(obj, ["map_lat", "est_lat", "latitude"])
        lon_k = _find_key(obj, ["map_lon", "est_lon", "longitude"])
        if lat_k and lon_k:
            est_lat = float(obj[lat_k])
            est_lon = float(obj[lon_k])

    return lat_grid, lon_grid, pdf_grid, est_lat, est_lon

def plot_bisl_plotly(
    loc_prefix="fk_inputs/loc/my_eventset-ev0",
    truth_lat=None, truth_lon=None,
    title="BISL posterior + estimate vs ground truth",
    zoom=8.5,
    prob_threshold=0.02,   # keep only probabilities above this fraction of max (speed)
    radius=18              # Densitymapbox smoothing
):
    fn, obj = load_bisl_json(loc_prefix)
    lat_grid, lon_grid, pdf_grid, est_lat, est_lon = extract_bisl_grid_and_point(obj)

    # Flatten for Plotly density map
    lat = lat_grid.ravel()
    lon = lon_grid.ravel()
    z   = pdf_grid.ravel().astype(float)

    # Normalize z to [0,1] for stable rendering
    zmax = np.nanmax(z)
    if not np.isfinite(zmax) or zmax <= 0:
        raise ValueError("Posterior grid has non-positive/invalid values.")
    z = z / zmax

    # Drop low-probability points for speed
    mask = np.isfinite(z) & (z >= prob_threshold)
    lat_h, lon_h, z_h = lat[mask], lon[mask], z[mask]

    # Center map: prefer estimate, else truth, else grid center
    if est_lat is not None and est_lon is not None:
        center_lat, center_lon = est_lat, est_lon
    elif truth_lat is not None and truth_lon is not None:
        center_lat, center_lon = truth_lat, truth_lon
    else:
        center_lat = float(np.nanmean(lat_grid))
        center_lon = float(np.nanmean(lon_grid))

    fig = go.Figure()

    # Posterior heatmap
    fig.add_trace(go.Densitymapbox(
        lat=lat_h,
        lon=lon_h,
        z=z_h,
        radius=radius,
        opacity=0.55,
        name="BISL posterior",
        hoverinfo="skip"
    ))

    # Estimated location (MAP)
    if est_lat is not None and est_lon is not None:
        fig.add_trace(go.Scattermapbox(
            lat=[est_lat],
            lon=[est_lon],
            mode="markers",
            marker=dict(size=16, color="yellow"),
            name="BISL estimate (MAP)",
            hovertemplate="BISL MAP<br>Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra></extra>"
        ))

    # Ground truth
    if truth_lat is not None and truth_lon is not None:
        fig.add_trace(go.Scattermapbox(
            lat=[truth_lat],
            lon=[truth_lon],
            mode="markers",
            marker=dict(size=16, color="lime"),
            name="Ground truth",
            hovertemplate="Ground truth<br>Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra></extra>"
        ))

    # Layout
    fig.update_layout(
        title=f"{title}<br><sup>Loaded: {fn}</sup>",
        mapbox=dict(
            style="open-street-map",   # token-free, reliable
            center=dict(lat=center_lat, lon=center_lon),
            zoom=zoom,
        ),
        height=750,
        width=950,
        legend=dict(
            x=0.98, y=0.98,
            xanchor="right", yanchor="top",
            bgcolor="rgba(255,255,255,0.8)"
        )
    )

    return fig

# --- USE IT ---
# Put your ground truth here:
truth_lat = 46.8523
truth_lon = -121.7603

fig = plot_bisl_plotly(
    loc_prefix="fk_inputs/loc/my_eventset-ev0",
    truth_lat=truth_lat,
    truth_lon=truth_lon,
    zoom=9.0,
    prob_threshold=0.02,
    radius=18,
    title="Infrasound BISL localization vs ground truth"
)
fig.show()


FileNotFoundError: No BISL JSON found for prefix 'fk_inputs/loc/my_eventset-ev0'. Try: ls fk_inputs/loc/my_eventset-ev0*

In [7]:
!ls fk_inputs/loc/my_eventset-ev0*

fk_inputs/loc/my_eventset-ev0.loc.json


In [5]:
!ls


comparing_to_pnsn_catalog.ipynb
comparing_with_pnsn_catalog.ipynb
fk_inputs
high_snr_traces.mseed
maps_html
plotting_commonly_detected_events.ipynb
plotting_the_infrasound_location_with_origianl_one.ipynb
stations.json
testing_location_using_enveloc.ipynb
testing_surface_event_pickers.ipynb
Untitled.ipynb
validating_enveloc_locations_for_esec_events.ipynb
visualizing_commonly_detected_events.py.ipynb
visualizing_infrasound_data_for_mt_rainier_surface_events.ipynb
walking_through_generate_common_events.ipynb
waveforms


In [11]:
import json
import numpy as np
import plotly.graph_objects as go
from math import radians, sin, cos, asin, sqrt

LOC_JSON = "fk_inputs/loc/my_eventset-ev0.loc.json"

# ---- helpers ----
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*asin(sqrt(a))

def find_array_key(d, candidates):
    """Return first key in dict d whose lowercase contains any candidate substring."""
    keys = list(d.keys())
    for c in candidates:
        c = c.lower()
        for k in keys:
            if c in k.lower():
                return k
    return None

def extract_bisl_fields(obj):
    """
    Robustly extract:
      lat_grid (2D), lon_grid (2D), pdf_grid (2D), map_lat, map_lon
    from InfraPy BISL .loc.json (structure can vary across versions).
    """
    # 1) Find MAP estimate
    map_lat = map_lon = None
    map_key = find_array_key(obj, ["map"])
    if map_key and isinstance(obj[map_key], dict):
        m = obj[map_key]
        lat_k = find_array_key(m, ["lat"])
        lon_k = find_array_key(m, ["lon"])
        if lat_k and lon_k:
            map_lat = float(m[lat_k])
            map_lon = float(m[lon_k])

    # 2) Find posterior grid + grids (might be top-level or nested)
    def try_extract(d):
        lat_k = find_array_key(d, ["lat_grid", "latitude_grid", "latgrid"])
        lon_k = find_array_key(d, ["lon_grid", "longitude_grid", "longrid"])
        pdf_k = find_array_key(d, ["pdf", "posterior", "prob", "likelihood"])
        if lat_k and lon_k and pdf_k:
            lat = np.array(d[lat_k], dtype=float)
            lon = np.array(d[lon_k], dtype=float)
            pdf = np.array(d[pdf_k], dtype=float)
            return lat, lon, pdf
        return None

    out = try_extract(obj)
    if out is None:
        # Search nested dicts
        for k, v in obj.items():
            if isinstance(v, dict):
                out = try_extract(v)
                if out is not None:
                    break

    if out is None:
        # If it still fails, print keys to see exact structure
        raise KeyError(
            "Couldn't auto-find lat/lon/pdf grids in the loc.json. "
            "Run: print(obj.keys()) and paste them."
        )

    lat_grid, lon_grid, pdf_grid = out
    return lat_grid, lon_grid, pdf_grid, map_lat, map_lon

# ---- load ----
with open(LOC_JSON, "r") as f:
    obj = json.load(f)

lat_grid, lon_grid, pdf_grid, map_lat, map_lon = extract_bisl_fields(obj)

print("Loaded:", LOC_JSON)
print("Grid shapes:", lat_grid.shape, lon_grid.shape, pdf_grid.shape)
print("MAP (if found):", map_lat, map_lon)

# ---- set your ground truth here ----
truth_lat = 46.8523
truth_lon = -121.7603

# ---- prep grid for plotly density layer ----
lat = lat_grid.ravel()
lon = lon_grid.ravel()
z   = pdf_grid.ravel()

# Normalize for stable visualization
zmax = np.nanmax(z)
z = z / zmax if zmax > 0 else z

# Speed: keep only higher-probability mass
threshold = 0.02   # adjust: 0.01 shows more area, 0.05 shows tighter blob
mask = np.isfinite(z) & (z >= threshold)

lat_h, lon_h, z_h = lat[mask], lon[mask], z[mask]

# ---- build figure ----
fig = go.Figure()

# Posterior heatmap
fig.add_trace(go.Densitymapbox(
    lat=lat_h,
    lon=lon_h,
    z=z_h,
    radius=18,      # smoothing (10–30)
    opacity=0.55,
    name="BISL posterior",
    hoverinfo="skip"
))

# BISL MAP point
if map_lat is not None and map_lon is not None:
    fig.add_trace(go.Scattermapbox(
        lat=[map_lat],
        lon=[map_lon],
        mode="markers",
        marker=dict(size=16, color="yellow"),
        name="BISL MAP",
        hovertemplate="BISL MAP<br>Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra></extra>"
    ))

# Ground truth point
fig.add_trace(go.Scattermapbox(
    lat=[truth_lat],
    lon=[truth_lon],
    mode="markers",
    marker=dict(size=16, color="lime"),
    name="Ground truth",
    hovertemplate="Ground truth<br>Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra></extra>"
))

# Center map
if map_lat is not None and map_lon is not None:
    center_lat, center_lon = map_lat, map_lon
else:
    center_lat, center_lon = truth_lat, truth_lon

fig.update_layout(
    title="BISL posterior heatmap + MAP vs ground truth",
    mapbox=dict(
        style="open-street-map",   # token-free
        center=dict(lat=center_lat, lon=center_lon),
        zoom=9.0
    ),
    height=750,
    width=950,
    legend=dict(
        x=0.98, y=0.98,
        xanchor="right", yanchor="top",
        bgcolor="rgba(255,255,255,0.8)"
    )
)

fig.show()

# ---- print error ----
if map_lat is not None and map_lon is not None:
    err_km = haversine_km(map_lat, map_lon, truth_lat, truth_lon)
    print(f"Great-circle error (MAP vs truth): {err_km:.2f} km")
else:
    print("MAP point not found in loc.json (heatmap is still plotted).")


KeyError: "Couldn't auto-find lat/lon/pdf grids in the loc.json. Run: print(obj.keys()) and paste them."

In [12]:
import json
import numpy as np

LOC_JSON = "fk_inputs/loc/my_eventset-ev0.loc.json"

with open(LOC_JSON, "r") as f:
    obj = json.load(f)

def find_2d_numeric_arrays(d, prefix=""):
    """
    Recursively find arrays that can be parsed as numeric 2D arrays.
    Returns list of (path, array).
    """
    found = []
    if isinstance(d, dict):
        for k, v in d.items():
            found += find_2d_numeric_arrays(v, prefix + k + ".")
    elif isinstance(d, list):
        # try convert to numpy array
        try:
            arr = np.array(d, dtype=float)
            if arr.ndim == 2 and arr.shape[0] > 5 and arr.shape[1] > 5:
                found.append((prefix[:-1], arr))
        except Exception:
            pass
    return found

arrays = find_2d_numeric_arrays(obj)

print("Found 2D numeric arrays:")
for path, arr in arrays:
    print(f"  {path:50s} shape={arr.shape} min={np.nanmin(arr):.3g} max={np.nanmax(arr):.3g}")

# Heuristic:
# - lat/lon grids often have ranges like lat ~ [45..49], lon ~ [-124..-120]
# - pdf grid often has small positive numbers and/or sums to ~1 after normalization
lat_grid = lon_grid = pdf_grid = None

for path, arr in arrays:
    mn, mx = float(np.nanmin(arr)), float(np.nanmax(arr))
    # longitude grid candidate
    if mn < -100 and mx < 0:
        lon_grid = arr
        lon_path = path
    # latitude grid candidate
    if mn > 0 and mx < 90:
        # but avoid grabbing pdf (pdf max usually << 1 or could be small)
        if mx > 5:  # lat will be ~ 46–48, pdf won't be >5
            lat_grid = arr
            lat_path = path

# pdf grid: same shape as lat/lon, positive, typically <=1-ish (but could be tiny)
if lat_grid is not None and lon_grid is not None:
    target_shape = lat_grid.shape
    for path, arr in arrays:
        if arr.shape == target_shape:
            mn, mx = float(np.nanmin(arr)), float(np.nanmax(arr))
            if mn >= 0 and mx <= 1.0:  # common
                pdf_grid = arr
                pdf_path = path
                break
    # fallback: pick the remaining array with same shape and nonnegative
    if pdf_grid is None:
        for path, arr in arrays:
            if arr.shape == target_shape:
                mn, mx = float(np.nanmin(arr)), float(np.nanmax(arr))
                if mn >= 0 and path not in [lat_path, lon_path]:
                    pdf_grid = arr
                    pdf_path = path
                    break

print("\nChosen:")
print(" lat:", lat_path if lat_grid is not None else None, lat_grid.shape if lat_grid is not None else None)
print(" lon:", lon_path if lon_grid is not None else None, lon_grid.shape if lon_grid is not None else None)
print(" pdf:", pdf_path if pdf_grid is not None else None, pdf_grid.shape if pdf_grid is not None else None)

# MAP point: search for any dict with keys containing lat/lon and numeric values
map_lat = map_lon = None
def find_map_point(d):
    global map_lat, map_lon
    if isinstance(d, dict):
        keys = [k.lower() for k in d.keys()]
        if any("lat" in k for k in keys) and any("lon" in k for k in keys):
            # try extract
            for k, v in d.items():
                if "lat" in k.lower():
                    try: map_lat = float(v)
                    except: pass
                if "lon" in k.lower():
                    try: map_lon = float(v)
                    except: pass
            if map_lat is not None and map_lon is not None:
                return True
        for v in d.values():
            if find_map_point(v):
                return True
    return False

find_map_point(obj)
print("\nMAP guess:", map_lat, map_lon)


Found 2D numeric arrays:

Chosen:
 lat: None None
 lon: None None
 pdf: None None

MAP guess: 46.94287104450641 -121.71598801147002


In [13]:
import plotly.graph_objects as go
import numpy as np
from math import radians, sin, cos, asin, sqrt

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*asin(sqrt(a))

# --- set truth ---
truth_lat, truth_lon = 46.8523, -121.7603  # change to your ground truth

# Flatten
lat = lat_grid.ravel()
lon = lon_grid.ravel()
z   = pdf_grid.ravel()

# normalize + threshold for speed
z = z / np.nanmax(z)
mask = np.isfinite(z) & (z >= 0.02)

fig = go.Figure()

fig.add_trace(go.Densitymapbox(
    lat=lat[mask],
    lon=lon[mask],
    z=z[mask],
    radius=18,
    opacity=0.55,
    name="BISL posterior",
    hoverinfo="skip"
))

if map_lat is not None and map_lon is not None:
    fig.add_trace(go.Scattermapbox(
        lat=[map_lat], lon=[map_lon],
        mode="markers",
        marker=dict(size=16, color="yellow"),
        name="BISL MAP"
    ))

fig.add_trace(go.Scattermapbox(
    lat=[truth_lat], lon=[truth_lon],
    mode="markers",
    marker=dict(size=16, color="lime"),
    name="Ground truth"
))

center_lat = map_lat if map_lat is not None else truth_lat
center_lon = map_lon if map_lon is not None else truth_lon

fig.update_layout(
    title="BISL posterior + MAP vs ground truth",
    mapbox=dict(style="open-street-map", center=dict(lat=center_lat, lon=center_lon), zoom=9.0),
    height=750, width=950
)

fig.show()

if map_lat is not None and map_lon is not None:
    print("Error (km):", haversine_km(map_lat, map_lon, truth_lat, truth_lon))


AttributeError: 'NoneType' object has no attribute 'ravel'

In [32]:
import json
import numpy as np
import plotly.graph_objects as go
from math import radians, sin, cos, asin, sqrt

LOC_JSON = "fk_inputs/loc/my_eventset-ev0.loc.json"

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

with open(LOC_JSON, "r") as f:
    obj = json.load(f)

# --- MAP estimate from BISL ---
map_lat = float(obj["lat_MaP"])
map_lon = float(obj["lon_MaP"])

# --- spatial posterior as scattered points ---
lat_flat, lon_flat, pdf_flat = obj["spatial_pdf"]
lat_flat = np.array(lat_flat, dtype=float)
lon_flat = np.array(lon_flat, dtype=float)
z = np.array(pdf_flat, dtype=float)

print("Lengths:", len(lat_flat), len(lon_flat), len(z))
print("MAP:", map_lat, map_lon)

# --- Your ground truth ---
truth_lat = 46.8435   # change
truth_lon = -121.7478 # change

# Normalize z for nicer plotting
z = z / np.nanmax(z)

# Speed/clarity: keep only higher-probability points
threshold = 0.0000001   # tune: 0.01 broader, 0.05 tighter
mask = np.isfinite(z) #& (z >= threshold)

fig = go.Figure()

# Posterior heatmap
fig.add_trace(go.Densitymapbox(
    lat=lat_flat[mask],
    lon=lon_flat[mask],
    z=z[mask],
    radius= 50,
    opacity=0.85,
    name="BISL posterior",
    hoverinfo="skip"
))

# BISL MAP point
fig.add_trace(go.Scattermapbox(
    lat=[map_lat],
    lon=[map_lon],
    mode="markers",
    marker=dict(size=16, color="yellow"),
    name="BISL MAP",
    hovertemplate="BISL MAP<br>Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra></extra>"
))

# Ground truth point
fig.add_trace(go.Scattermapbox(
    lat=[truth_lat],
    lon=[truth_lon],
    mode="markers",
    marker=dict(size=16, color="lime"),
    name="Ground truth",
    hovertemplate="Ground truth<br>Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra></extra>"
))

# Line between them
fig.add_trace(go.Scattermapbox(
    lat=[map_lat, truth_lat],
    lon=[map_lon, truth_lon],
    mode="lines",
    line=dict(width=3, color="black"),
    name="Error vector",
    hoverinfo="skip"
))


fig.add_trace(go.Scattermapbox(
    lat=lat_flat[mask],
    lon=lon_flat[mask],
    mode="markers",
    marker=dict(
        size=6,
        color=z[mask],
        colorscale="Plasma",
        cmin=0,
        cmax=1,
        opacity=0.6,
        colorbar=dict(title="Posterior")
    ),
    name="Posterior (points)",
    hoverinfo="skip"
))


fig.update_layout(
    title="BISL posterior + MAP vs ground truth",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=map_lat, lon=map_lon),
        zoom=9.0
    ),
    height=750,
    width=950,
    legend=dict(
        x=0.98, y=0.98,
        xanchor="right", yanchor="top",
        bgcolor="rgba(255,255,255,0.8)"
    )
)

fig.show()

print(f"Error (km): {haversine_km(map_lat, map_lon, truth_lat, truth_lon):.2f}")


Lengths: 6400 6400 6400
MAP: 46.876077565857784 -121.84665677180244


Error (km): 8.34


In [33]:
print("Total grid points:", len(z))
print("Points kept after threshold:", mask.sum())
print("z min/max (norm):", float(np.nanmin(z)), float(np.nanmax(z)))


Total grid points: 6400
Points kept after threshold: 6400
z min/max (norm): 1.695439056476503e-80 1.0


In [34]:
import numpy as np

# lat_flat, lon_flat, z already loaded from spatial_pdf, and normalized (0..1)
mask = np.isfinite(z) & (z >= 0.0)   # no threshold for debugging

print("MAP:", map_lat, map_lon)
print("Truth:", truth_lat, truth_lon)

print("ALL posterior lat min/max:", float(lat_flat.min()), float(lat_flat.max()))
print("ALL posterior lon min/max:", float(lon_flat.min()), float(lon_flat.max()))

top = z >= np.quantile(z, 0.995)   # top 0.5% most probable points
print("TOP posterior count:", int(top.sum()))
print("TOP posterior lat min/max:", float(lat_flat[top].min()), float(lat_flat[top].max()))
print("TOP posterior lon min/max:", float(lon_flat[top].min()), float(lon_flat[top].max()))



MAP: 46.876077565857784 -121.84665677180244
Truth: 46.8435 -121.7478
ALL posterior lat min/max: -122.04567689720959 -119.42524524601545
ALL posterior lon min/max: 45.76046289648195 47.55910695527157
TOP posterior count: 32
TOP posterior lat min/max: -121.87982679270362 -121.64763664639528
TOP posterior lon min/max: 46.8305422732302 46.98991579742675


In [ ]:
# Posterior heatmap
fig.add_trace(go.Densitymapbox(
    lat=lat_flat[mask],
    lon=lon_flat[mask],
    z=z[mask],
    radius= 50,
    opacity=0.85,
    name="BISL posterior",
    hoverinfo="skip"
))

In [13]:
import json
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

LOC_JSON = "infrapy_runs/ev_0000/loc/my_eventset-ev0.loc.json"
with open(LOC_JSON) as f:
    obj = json.load(f)

map_lat = float(obj["lat_MaP"])
map_lon = float(obj["lon_MaP"])

# NOTE: spatial_pdf is [lon_flat, lat_flat, pdf_flat] in your file
lon_flat, lat_flat, pdf_flat = obj["spatial_pdf"]
lon_flat = np.array(lon_flat, dtype=float)
lat_flat = np.array(lat_flat, dtype=float)
z = np.array(pdf_flat, dtype=float)

# normalize
z = z / np.nanmax(z)

# keep top 1% for visibility + speed
top = z >= np.quantile(z, 0.99)

# bounds from the posterior (correctly using lat_flat/lon_flat)
south, north = float(lat_flat[top].min()), float(lat_flat[top].max())
west,  east  = float(lon_flat[top].min()), float(lon_flat[top].max())

truth_lat, truth_lon =0, 0 # your truth

fig = go.Figure()

# Posterior as BIG colored points (you will see it)
fig.add_trace(go.Scattermapbox(
    lat=lat_flat[top],
    lon=lon_flat[top],
    mode="markers",
    marker=dict(
        size=22,
        color=z[top],
        colorscale="Plasma",
        cmin=0, cmax=1,
        opacity=0.85,
        colorbar=dict(title="Posterior")
    ),
    name="Posterior (top 1%)",
    hoverinfo="skip"
))

fig.add_trace(go.Scattermapbox(
    lat=[map_lat], lon=[map_lon],
    mode="markers",
    marker=dict(size=16, color="yellow"),
    name="BISL MAP"
))

fig.add_trace(go.Scattermapbox(
    lat=[truth_lat], lon=[truth_lon],
    mode="markers",
    marker=dict(size=16, color="lime"),
    name="Ground truth"
))

fig.update_layout(
    title="BISL posterior + MAP vs ground truth",
    mapbox=dict(
        style="open-street-map",
        bounds=dict(west=west, east=east, south=south, north=north)
    ),
    height=750, width=950
)

fig.show()


In [14]:
fig = go.Figure()

fig.add_trace(go.Densitymapbox(
    lat=lat_flat,
    lon=lon_flat,
    z=z,
    radius=30,
    opacity=0.7,
    name="BISL posterior",
    hoverinfo="skip"
))

fig.add_trace(go.Scattermapbox(
    lat=[map_lat], lon=[map_lon],
    mode="markers",
    marker=dict(size=16, color="yellow"),
    name="BISL MAP"
))

""""
fig.add_trace(go.Scattermapbox(
    lat=[truth_lat], lon=[truth_lon],
    mode="markers",
    marker=dict(size=16, color="lime"),
    name="Ground truth"
))
"""

fig.update_layout(
    title="BISL posterior heatmap + MAP vs ground truth",
    mapbox=dict(style="open-street-map", center=dict(lat=map_lat, lon=map_lon), zoom=9.0),
    height=750, width=950
)
fig.show()


In [52]:
import re
import numpy as np
import plotly.graph_objects as go

sensor_text = """
CC MIRR AttribDict({'latitude': 46.800557, 'longitude': -121.837642, 'elevation': 1653.6})
CC MIRR AttribDict({'latitude': 46.800646, 'longitude': -121.837753, 'elevation': 1653.6})
CC MIRR AttribDict({'latitude': 46.800691, 'longitude': -121.837567, 'elevation': 1653.6})
CC MIRR AttribDict({'latitude': 46.800526, 'longitude': -121.837807, 'elevation': 1653.6})
CC PR03 AttribDict({'latitude': 46.903346, 'longitude': -122.032643, 'elevation': 529.0})
CC PR03 AttribDict({'latitude': 46.903627, 'longitude': -122.03269, 'elevation': 531.0})
CC PR03 AttribDict({'latitude': 46.903438, 'longitude': -122.032271, 'elevation': 532.0})
CC PR04 AttribDict({'latitude': 46.93003, 'longitude': -121.988387, 'elevation': 911.0})
CC PR04 AttribDict({'latitude': 46.929746, 'longitude': -121.988485, 'elevation': 914.1})
CC PR04 AttribDict({'latitude': 46.929694, 'longitude': -121.988976, 'elevation': 912.7})
CC PR05 AttribDict({'latitude': 46.841732, 'longitude': -121.948846, 'elevation': 1553.0})
CC PR05 AttribDict({'latitude': 46.841658, 'longitude': -121.948748, 'elevation': 1553.0})
CC PR05 AttribDict({'latitude': 46.841724, 'longitude': -121.949017, 'elevation': 1553.0})
"""

# Parse lines like:
# CC PR03 AttribDict({'latitude': 46.903346, 'longitude': -122.032643, 'elevation': 529.0})
pattern = re.compile(
    r"^\s*(\w+)\s+(\w+)\s+AttribDict\(\{'latitude':\s*([-\d.]+),\s*'longitude':\s*([-\d.]+),\s*'elevation':\s*([-\d.]+)\}\)\s*$"
)

by_station = {}
for line in sensor_text.strip().splitlines():
    m = pattern.match(line)
    if not m:
        continue
    net, sta, lat, lon, elev = m.groups()
    key = f"{net}.{sta}"
    by_station.setdefault(key, {"lats": [], "lons": [], "elevs": []})
    by_station[key]["lats"].append(float(lat))
    by_station[key]["lons"].append(float(lon))
    by_station[key]["elevs"].append(float(elev))

sensor_names = sorted(by_station.keys())
sensor_lats  = [float(np.mean(by_station[k]["lats"]))  for k in sensor_names]
sensor_lons  = [float(np.mean(by_station[k]["lons"]))  for k in sensor_names]
sensor_elevs = [float(np.mean(by_station[k]["elevs"])) for k in sensor_names]



fig.add_trace(go.Scattermapbox(
    lat=sensor_lats,
    lon=sensor_lons,
    mode="markers+text",          # or "markers" if you want no labels
    text=sensor_names,
    textposition="top center",
    marker=dict(
        size=14,
        color="black",
        symbol="triangle-up"
    ),
    customdata=sensor_elevs,
    hovertemplate="<b>%{text}</b><br>lat=%{lat:.5f}<br>lon=%{lon:.5f}<br>elev=%{customdata:.1f} m<extra></extra>",
    name="Infrasound sensors"
))
